# Embeddings in Google Colab berechnen

1. Laufzeit → Laufzeittyp ändern → **T4 GPU** → Speichern
2. Laufzeit → **Alle ausführen**
3. Bei der zweiten Zelle erscheint „Dateien auswählen“: `cache/colab_input.jsonl` aus dem Repository-Ordner wählen
4. Am Ende wird `embeddings.zip` heruntergeladen. Den Inhalt in den Ordner `cache/` des Repositorys entpacken.

Die Datei `colab_input.jsonl` erzeugt `scripts/export_for_colab.py`. Sie enthält nur die drei Stichproben (je 10.000 Texte, Seed 42) und die Namen der Cache-Dateien, die `run_experiments.py` erwartet.

In [ ]:
!pip install -q sentence-transformers
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "keine – bitte T4 GPU einstellen")

In [ ]:
import json
from google.colab import files
uploaded = files.upload()  # cache/colab_input.jsonl auswählen
name = next(iter(uploaded))
jobs = [json.loads(line) for line in uploaded[name].decode("utf-8").splitlines() if line.strip()]
for j in jobs:
    print(j["corpus"], len(j["texts"]), j["cache_file"])

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(jobs[0]["model"], device="cuda" if torch.cuda.is_available() else "cpu")
log = {}
for j in jobs:
    model.max_seq_length = j["max_seq_length"]
    t0 = time.perf_counter()
    E = model.encode([j["prefix"] + t for t in j["texts"]], batch_size=64, show_progress_bar=True,
                     normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    np.save(j["cache_file"], E)
    lengths = [len(ids) for ids in model.tokenizer([j["prefix"] + t for t in j["texts"]])["input_ids"]]
    log[j["corpus"]] = {"shape": list(E.shape), "seconds": round(time.perf_counter() - t0, 1),
                        "truncated": int(sum(l > j["max_seq_length"] for l in lengths)),
                        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    print(j["corpus"], log[j["corpus"]])
json.dump(log, open("embedding_log.json", "w"), indent=2)

In [ ]:
import zipfile
with zipfile.ZipFile("embeddings.zip", "w") as z:
    for j in jobs:
        z.write(j["cache_file"])
    z.write("embedding_log.json")
files.download("embeddings.zip")